### Loading SB County Rainfall Data

- Rain gauge data downloaded from the hydrology section of the Santa Barbara County website
- Need to determine the number of rain gauges we are using data from

#### Steps
- Clean data from 

In [1]:
# Load packages
import os
import sys
import numpy as np
import pandas as pd
import geopandas as gpd

import os
import sys

sys.path.append("../utils")

import config


In [2]:
file_list = [
    os.path.join(config.data_dir, "sb_rain_gauges", "155dailys.xls"),
    os.path.join(config.data_dir, "sb_rain_gauges", "156dailys.xls"),
    os.path.join(config.data_dir, "sb_rain_gauges", "196dailys.xls"),
    os.path.join(config.data_dir, "sb_rain_gauges", "197dailys.xls")
    # Add more files as needed
]
rain_gauge = os.path.join(
    config.data_dir,
    "sb_rain_gauges",
    "197dailys.xls"
)
rain_test = pd.read_excel(rain_gauge, header = 9)

In [3]:
for f in file_list:
    print(f)

/capstone/wildfire_prep/data/sb_rain_gauges/155dailys.xls
/capstone/wildfire_prep/data/sb_rain_gauges/156dailys.xls
/capstone/wildfire_prep/data/sb_rain_gauges/196dailys.xls
/capstone/wildfire_prep/data/sb_rain_gauges/197dailys.xls


In [4]:
test = pd.read_excel(rain_gauge, nrows = 8, usecols="A")
test

,County of Santa Barbara
0,Daily Rainfall Record - through 08-31-2024
1,NaN
2,#197 - Las Cruces
3,"Lat 34-30-27, Long 120-13-48, Elev 390ft"
4,NaN
5,Daily Rainfall (in inches) recorded as of 8am ...
6,"Codes: PR = Preliminary data, E = Estimated f..."
7,NaN


In [5]:
# Function to convert Degrees-Minutes-Seconds to Decimal Degrees
def dms_to_decimal(dms):
    degrees, minutes, seconds = map(int, dms.split('-'))
    return degrees + (minutes / 60) + (seconds / 3600)

In [6]:
# Creating `latitude`, `longitude`, and `elevation` columns
rain_test[['lat', 'long', 'elevation']] = (test['County of Santa Barbara'].dropna()
                                           .loc[lambda x: x.str.startswith('L')]
                                           .iloc[0]
                                           .split(', '))
                                           

# Removing text from `latitude`, `longitude`, and `elevation` columns
rain_test['lat'] = rain_test['lat'].apply(lambda x: x.replace('Lat ', '')).apply(dms_to_decimal)
rain_test['long'] = rain_test['long'].apply(lambda x: x.replace('Long ', '')).apply(dms_to_decimal)
rain_test['elevation'] = rain_test['elevation'].apply(lambda x: int(x.replace('Elev ', '').replace('ft', '')))
rain_test

,station id,water year,year,month,day,daily rain,code,lat,long,elevation
0,197,2013,2013,6,3,0.04,NaN,34.5075,120.23,390
1,197,2013,2013,6,9,0.04,NaN,34.5075,120.23,390
2,197,2013,2013,6,10,0.01,NaN,34.5075,120.23,390
3,197,2013,2013,6,11,0.01,NaN,34.5075,120.23,390
4,197,2013,2013,7,22,0.01,NaN,34.5075,120.23,390
...,...,...,...,...,...,...,...,...,...,...
552,197,2024,2024,5,5,0.08,NaN,34.5075,120.23,390
553,197,2024,2024,6,6,0.01,NaN,34.5075,120.23,390
554,197,2024,2024,6,8,0.01,NaN,34.5075,120.23,390
555,197,2024,2024,6,13,0.01,NaN,34.5075,120.23,390


In [7]:
def clean_rain_gauge_data(excel_file):
    #
    rain = pd.read_excel(excel_file, header = 9)

    #
    metadata = pd.read_excel(excel_file, nrows = 8, usecols="A")
    
    # Creating `latitude`, `longitude`, and `elevation` columns
    rain[['lat', 'long', 'elevation']] = (metadata['County of Santa Barbara'].dropna()
                                           .loc[lambda x: x.str.startswith('L')]
                                           .iloc[0]
                                           .split(', '))                                     

    # Removing text from `latitude`, `longitude`, and `elevation` columns
    rain['lat'] = rain['lat'].apply(lambda x: x.replace('Lat ', '')).apply(dms_to_decimal)
    rain['long'] = rain['long'].apply(lambda x: x.replace('Long ', '')).apply(dms_to_decimal)
    rain['elevation'] = rain['elevation'].apply(lambda x: int(x.replace('Elev ', '').replace('ft', '')))

    # Make `station id ` character instead of numeric
    rain['stationid'] = rain['stationid'].astype(str)

    return rain

In [8]:
# Function to process a list of files and append them to a single DataFrame
def process_rain_gauges(file_list):
    # Initialize an empty DataFrame to append data
    rain_data = pd.DataFrame()
    
    for file_path in file_list:
        print(f"Processing file: {file_path}")
        cleaned = clean_rain_gauge_data(file_path)
        rain_data = pd.concat([rain_data, cleaned], ignore_index=True)
    
    return rain_data


In [9]:
rain_data = process_rain_gauges(file_list)


Processing file: /capstone/wildfire_prep/data/sb_rain_gauges/155dailys.xls
Processing file: /capstone/wildfire_prep/data/sb_rain_gauges/156dailys.xls
Processing file: /capstone/wildfire_prep/data/sb_rain_gauges/196dailys.xls


KeyError: 'stationid'

In [10]:
rain_data['stationid']

NameError: name 'rain_data' is not defined

In [ ]:
clean_rain_gauge_data(rain_gauge)

,station id,water year,year,month,day,daily rain,code,lat,long,elevation
0,197,2013,2013,6,3,0.04,NaN,34.5075,120.23,390
1,197,2013,2013,6,9,0.04,NaN,34.5075,120.23,390
2,197,2013,2013,6,10,0.01,NaN,34.5075,120.23,390
3,197,2013,2013,6,11,0.01,NaN,34.5075,120.23,390
4,197,2013,2013,7,22,0.01,NaN,34.5075,120.23,390
...,...,...,...,...,...,...,...,...,...,...
552,197,2024,2024,5,5,0.08,NaN,34.5075,120.23,390
553,197,2024,2024,6,6,0.01,NaN,34.5075,120.23,390
554,197,2024,2024,6,8,0.01,NaN,34.5075,120.23,390
555,197,2024,2024,6,13,0.01,NaN,34.5075,120.23,390
